<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/04_transformers/bert_vs_sentence_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers sentence-transformers torch scikit-learn

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
sentences = [
    "Machine learning is transforming technology",
    "Artificial intelligence is changing the world",
    "I enjoy cooking food",
    "Natural language processing is a part of AI"
]

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModel.from_pretrained("bert-base-uncased")

In [22]:
bert_inputs = bert_tokenizer(
    sentences,
    return_tensors="pt",
    padding=True,
    truncation=True
)

In [23]:
with torch.no_grad():
    bert_outputs = bert_model(**bert_inputs)

In [24]:
token_embeddings = bert_outputs.last_hidden_state
attention_mask = bert_inputs["attention_mask"]

mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
bert_sentence_embeddings = (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

bert_sentence_embeddings.shape

torch.Size([4, 768])

In [25]:
bert_similarity = cosine_similarity(bert_sentence_embeddings.numpy())
bert_similarity

array([[1.0000001 , 0.8723478 , 0.53521866, 0.7210303 ],
       [0.8723478 , 0.9999999 , 0.508627  , 0.63726246],
       [0.53521866, 0.508627  , 0.9999996 , 0.47150266],
       [0.7210303 , 0.63726246, 0.47150266, 1.0000001 ]], dtype=float32)

In [ ]:
st_model = SentenceTransformer("all-MiniLM-L6-v2")

In [26]:
st_embeddings = st_model.encode(sentences)
st_embeddings.shape

(4, 384)

In [27]:
st_similarity = cosine_similarity(st_embeddings)
st_similarity

array([[0.99999994, 0.6296227 , 0.06609212, 0.36944965],
       [0.6296227 , 1.        , 0.06505713, 0.47657144],
       [0.06609212, 0.06505713, 0.9999999 , 0.12223791],
       [0.36944965, 0.47657144, 0.12223791, 0.9999997 ]], dtype=float32)

In [28]:
print("BERT Similarity (Sentence 1 vs Others):")
print(bert_similarity[0])

print("\nSentence Transformer Similarity (Sentence 1 vs Others):")
print(st_similarity[0])

BERT Similarity (Sentence 1 vs Others):
[1.0000001  0.8723478  0.53521866 0.7210303 ]

Sentence Transformer Similarity (Sentence 1 vs Others):
[0.99999994 0.6296227  0.06609212 0.36944965]


- This notebook compares raw BERT sentence embeddings with sentence-transformer embeddings.
- BERT embeddings are generated using mean pooling over token embeddings, while sentence-transformers use models fine-tuned specifically for sentence similarity tasks.
- The comparison shows that sentence-transformers generally produce more meaningful and consistent similarity scores for semantic tasks.
